# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [5]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: f:\UIUC\cs521\hw3\hw3\q3\linf_base.pt

  0%|          | 0.00/11.2M [00:00<?, ?B/s]
 23%|██▎       | 2.62M/11.2M [00:00<00:00, 22.7MB/s]
 56%|█████▌    | 6.29M/11.2M [00:00<00:00, 28.4MB/s]
 89%|████████▊ | 9.96M/11.2M [00:00<00:00, 31.5MB/s]
100%|██████████| 11.2M/11.2M [00:00<00:00, 30.6MB/s]
Using cookies from C:\Users\ASUS\.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=21cdc693-02a8-4bc4-af6e-f5da40b128b2
To: f:\UIUC\cs521\hw3\hw3\q3\cifar10_train_10k.pt

  0%|          | 0.00/30.8M [00:00<?, ?B/s]
  9%|▊         | 2.62M/30.8M [00:00<00:01, 22.6MB/s]
 20%|██        | 6.29M/30.8M [00:00<00:00, 28.5MB/s]
 34%|███▍      | 10.5M/30.8M [00:00<00:00, 31.9MB/s]
 48%|████▊     | 14.7M/30.8M [00:00<00:00, 34.5MB/s]
 61%|██████▏   | 

In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}
# EPS = {"linf": 4 / 255, "l2": 0.5, "l1": 12.0}
# EPS = {"linf": 2 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [2]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [3]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [4]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    else:
        delta = torch.zeros_like(x)
    adv_x = (x + delta).clamp(0, 1)

    for _ in range(steps):
        grad = input_grad(model, x, adv_x - x, y)
        adv_x = adv_x + alpha * grad.sign()
        adv_x = torch.clamp(adv_x, x - eps, x + eps)
        adv_x = torch.clamp(adv_x, 0, 1)
        adv_x = adv_x.clone().detach()
   
    return adv_x


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    if rand_init:
        delta = torch.randn_like(x)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta) * eps * r
    else:
        delta = torch.zeros_like(x)
    adv_x = (x + delta).clamp(0, 1)

    for _ in range(steps):
        grad = input_grad(model, x, adv_x - x, y)
        adv_x = adv_x + alpha * grad / (l2_norm(grad) + 1e-10)
        d = adv_x - x
        d = d * torch.clamp(eps / (l2_norm(d) + 1e-10), max=1.0)
        adv_x = x + clip_box(x, d)
        adv_x = adv_x.detach()
   
    return adv_x

In [5]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [6]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [7]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.5  l2 59.8  l1 23.0  union 23.0   (39 s)


{'clean': 78.7, 'linf': 48.5, 'l2': 59.8, 'l1': 23.0, 'union': 23.0}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [9]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()

    return F.cross_entropy(model(x_adv), y)

In [10]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.199 (16 s)
epoch 2: loss 1.178 (32 s)
epoch 3: loss 1.153 (49 s)
epoch 4: loss 1.120 (67 s)
epoch 5: loss 1.101 (83 s)
epoch 6: loss 1.071 (100 s)
epoch 7: loss 1.044 (117 s)
epoch 8: loss 1.025 (134 s)
epoch 9: loss 1.011 (152 s)
epoch 10: loss 1.005 (169 s)
clean 78.0  linf 44.5  l2 57.2  l1 19.2  union 19.1   (47 s)


{'clean': 78.0, 'linf': 44.5, 'l2': 57.2, 'l1': 19.2, 'union': 19.1}

In [11]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.898 (19 s)
epoch 2: loss 0.833 (37 s)
epoch 3: loss 0.791 (56 s)
epoch 4: loss 0.749 (75 s)
epoch 5: loss 0.710 (94 s)
epoch 6: loss 0.670 (112 s)
epoch 7: loss 0.639 (131 s)
epoch 8: loss 0.612 (150 s)
epoch 9: loss 0.589 (169 s)
epoch 10: loss 0.578 (188 s)
clean 82.9  linf 32.4  l2 60.3  l1 29.1  union 25.3   (51 s)


{'clean': 82.9, 'linf': 32.4, 'l2': 60.3, 'l1': 29.1, 'union': 25.3}

In [12]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.231 (20 s)
epoch 2: loss 0.839 (40 s)
epoch 3: loss 0.577 (60 s)
epoch 4: loss 0.446 (79 s)
epoch 5: loss 0.335 (99 s)
epoch 6: loss 0.278 (118 s)
epoch 7: loss 0.203 (138 s)
epoch 8: loss 0.180 (157 s)
epoch 9: loss 0.159 (176 s)
epoch 10: loss 0.154 (196 s)
clean 83.8  linf 0.0  l2 0.0  l1 26.2  union 0.0   (51 s)


{'clean': 83.8, 'linf': 0.0, 'l2': 0.0, 'l1': 26.2, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [15]:
def kl_per_sample(logits_p, logits_q):
    log_p = F.log_softmax(logits_p, dim=1)
    log_q = F.log_softmax(logits_q, dim=1)
    return (log_p.exp() * (log_p - log_q)).sum(dim=1)

In [22]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    adv_x_q = attack(model, x, y, "linf")
    adv_x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(adv_x_q)
    logits_r = model(adv_x_r)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y,reduction="none")
    adv = torch.maximum(ce_q, ce_r).mean()
    correct = logits_q.argmax(1) == y

    kl = kl_per_sample(logits_q[correct], logits_r[correct]).mean()
    ramp_loss = adv + lam * kl

    return ramp_loss


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    cosine = F.cosine_similarity(g_n.flatten(), g_a.flatten(), dim=0)
    if cosine > 0:
        return cosine * g_n
    else:
        return torch.zeros_like(g_n)

In [23]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.432 (45 s)
epoch 2: loss 1.337 (93 s)
epoch 3: loss 1.301 (138 s)
epoch 4: loss 1.273 (185 s)
epoch 5: loss 1.248 (229 s)
epoch 6: loss 1.218 (274 s)
epoch 7: loss 1.205 (322 s)
epoch 8: loss 1.185 (369 s)
epoch 9: loss 1.172 (415 s)
epoch 10: loss 1.168 (464 s)
clean 78.3  linf 41.6  l2 62.5  l1 45.1  union 39.5   (56 s)


{'clean': 78.3, 'linf': 41.6, 'l2': 62.5, 'l1': 45.1, 'union': 39.5}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [35]:
teacher = load_base_model()
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

In [43]:
def my_loss(model, x, y, lam=0.5, mu=1.0):
    # TODO: ramp_loss, but detach logits_q and teacher
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    adv = torch.maximum(ce_q, ce_r).mean()

    correct_q = logits_q.argmax(1) == y

    kl = kl_per_sample(logits_q[correct_q].detach(), logits_r[correct_q]).mean()

    with torch.no_grad():
        logits_t = teacher(x_q)
    distill = kl_per_sample(logits_t, logits_q).mean()

    return adv + lam * kl + mu * distill

In [37]:
model_ramp_s1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
test_model(model_ramp_s1)

epoch 1: loss 1.433 (38 s)
epoch 2: loss 1.328 (76 s)
epoch 3: loss 1.300 (114 s)
epoch 4: loss 1.271 (152 s)
epoch 5: loss 1.249 (192 s)
epoch 6: loss 1.228 (232 s)
epoch 7: loss 1.202 (272 s)
epoch 8: loss 1.183 (314 s)
epoch 9: loss 1.169 (355 s)
epoch 10: loss 1.172 (397 s)
clean 78.4  linf 41.9  l2 62.3  l1 45.5  union 39.9   (53 s)


{'clean': 78.4, 'linf': 41.9, 'l2': 62.3, 'l1': 45.5, 'union': 39.9}

In [40]:
model_mine_s1 = adv_train(load_base_model(), my_loss, gp_beta=0.5, seed=1)
test_model(model_mine_s1)

epoch 1: loss 1.486 (36 s)
epoch 2: loss 1.374 (73 s)
epoch 3: loss 1.340 (110 s)
epoch 4: loss 1.308 (149 s)
epoch 5: loss 1.282 (190 s)
epoch 6: loss 1.260 (231 s)
epoch 7: loss 1.234 (271 s)
epoch 8: loss 1.215 (312 s)
epoch 9: loss 1.200 (355 s)
epoch 10: loss 1.204 (397 s)
clean 78.5  linf 42.9  l2 62.6  l1 45.8  union 40.0   (50 s)


{'clean': 78.5, 'linf': 42.9, 'l2': 62.6, 'l1': 45.8, 'union': 40.0}

In [41]:
model_ramp_s0 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=0)
test_model(model_ramp_s0)

epoch 1: loss 1.432 (41 s)
epoch 2: loss 1.337 (82 s)
epoch 3: loss 1.301 (123 s)
epoch 4: loss 1.273 (164 s)
epoch 5: loss 1.249 (206 s)
epoch 6: loss 1.218 (249 s)
epoch 7: loss 1.205 (290 s)
epoch 8: loss 1.186 (332 s)
epoch 9: loss 1.172 (374 s)
epoch 10: loss 1.168 (418 s)
clean 78.5  linf 41.9  l2 62.5  l1 45.3  union 40.0   (51 s)


{'clean': 78.5, 'linf': 41.9, 'l2': 62.5, 'l1': 45.3, 'union': 40.0}

In [42]:
model_mine_s0 = adv_train(load_base_model(), my_loss, gp_beta=0.5, seed=0)
test_model(model_mine_s0)

epoch 1: loss 1.485 (43 s)
epoch 2: loss 1.382 (89 s)
epoch 3: loss 1.343 (133 s)
epoch 4: loss 1.309 (178 s)
epoch 5: loss 1.282 (225 s)
epoch 6: loss 1.253 (270 s)
epoch 7: loss 1.236 (314 s)
epoch 8: loss 1.218 (360 s)
epoch 9: loss 1.205 (405 s)
epoch 10: loss 1.202 (450 s)
clean 78.7  linf 43.0  l2 62.6  l1 45.5  union 40.3   (53 s)


{'clean': 78.7, 'linf': 43.0, 'l2': 62.6, 'l1': 45.5, 'union': 40.3}

In [46]:
def my_loss_v2(model, x, y, lam=0.5, mu=1.0):
    # ramp_loss, but detach logits_q and teacher and teacher distillation only on images that are l1-correct but linf-wrong
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    adv = torch.maximum(ce_q, ce_r).mean()

    pred_q = logits_q.argmax(1)
    pred_r = logits_r.argmax(1)

    correct_q = pred_q == y
    pair = kl_per_sample(logits_q[correct_q].detach(), logits_r[correct_q]).mean()

    zero = torch.zeros((), device=x.device)
    with torch.no_grad():
        logits_t = teacher(x_q)
    disagree = (pred_r == y) & (pred_q != y) & (logits_t.argmax(1) == y)
    if disagree.any():
        distill = kl_per_sample(logits_t[disagree], logits_q[disagree]).mean()
    else:
        distill = zero

    return adv + lam * pair + mu * distill

In [47]:
model_mine2_s1 = adv_train(load_base_model(), my_loss_v2, gp_beta=0.5, seed=1)
test_model(model_mine2_s1)

epoch 1: loss 1.693 (37 s)
epoch 2: loss 1.595 (75 s)
epoch 3: loss 1.561 (114 s)
epoch 4: loss 1.513 (154 s)
epoch 5: loss 1.559 (193 s)
epoch 6: loss 1.514 (234 s)
epoch 7: loss 1.408 (274 s)
epoch 8: loss 1.356 (316 s)
epoch 9: loss 1.326 (357 s)
epoch 10: loss 1.323 (399 s)
clean 76.2  linf 44.7  l2 61.3  l1 45.8  union 41.4   (51 s)


{'clean': 76.2, 'linf': 44.7, 'l2': 61.3, 'l1': 45.8, 'union': 41.4}

In [48]:
model_mine2_s0 = adv_train(load_base_model(), my_loss_v2, gp_beta=0.5, seed=0)
test_model(model_mine2_s0)

epoch 1: loss 1.650 (43 s)
epoch 2: loss 1.763 (85 s)
epoch 3: loss 1.624 (128 s)
epoch 4: loss 1.520 (170 s)
epoch 5: loss 1.527 (213 s)
epoch 6: loss 1.437 (255 s)
epoch 7: loss 1.387 (298 s)
epoch 8: loss 1.364 (340 s)
epoch 9: loss 1.325 (383 s)
epoch 10: loss 1.319 (426 s)
clean 76.4  linf 44.3  l2 62.0  l1 45.5  union 41.3   (53 s)


{'clean': 76.4, 'linf': 44.3, 'l2': 62.0, 'l1': 45.5, 'union': 41.3}

## Q3 Ablation

In [49]:
def my_loss_v2_no_detach(model, x, y, lam=0.5, mu=1.0):
    # ramp_loss, but detach logits_q and teacher and teacher distillation only on images that are l1-correct but linf-wrong
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    adv = torch.maximum(ce_q, ce_r).mean()

    pred_q = logits_q.argmax(1)
    pred_r = logits_r.argmax(1)

    correct_q = pred_q == y
    pair = kl_per_sample(logits_q[correct_q], logits_r[correct_q]).mean()

    zero = torch.zeros((), device=x.device)
    with torch.no_grad():
        t_logits = teacher(x_q)
    disagree = (pred_r == y) & (pred_q != y) & (t_logits.argmax(1) == y)
    if disagree.any():
        distill = kl_per_sample(t_logits[disagree], logits_q[disagree]).mean()
    else:
        distill = zero

    return adv + lam * pair + mu * distill

In [50]:
model_mine2_nd_s1 = adv_train(load_base_model(), my_loss_v2, gp_beta=0.5, seed=1)
test_model(model_mine2_nd_s1)

epoch 1: loss 1.668 (37 s)
epoch 2: loss 1.755 (77 s)
epoch 3: loss 1.642 (119 s)
epoch 4: loss 1.481 (159 s)
epoch 5: loss 1.467 (199 s)
epoch 6: loss 1.433 (240 s)
epoch 7: loss 1.388 (280 s)
epoch 8: loss 1.367 (320 s)
epoch 9: loss 1.322 (361 s)
epoch 10: loss 1.325 (405 s)
clean 77.2  linf 44.6  l2 62.5  l1 45.7  union 41.5   (50 s)


{'clean': 77.2, 'linf': 44.6, 'l2': 62.5, 'l1': 45.7, 'union': 41.5}

In [51]:
model_mine2_nd_s0 = adv_train(load_base_model(), my_loss_v2, gp_beta=0.5, seed=0)
test_model(model_mine2_nd_s0)

epoch 1: loss 1.595 (43 s)
epoch 2: loss 1.576 (89 s)
epoch 3: loss 1.611 (131 s)
epoch 4: loss 1.526 (174 s)
epoch 5: loss 1.485 (219 s)
epoch 6: loss 1.428 (262 s)
epoch 7: loss 1.380 (307 s)
epoch 8: loss 1.331 (350 s)
epoch 9: loss 1.312 (393 s)
epoch 10: loss 1.312 (437 s)
clean 76.9  linf 44.8  l2 62.2  l1 46.2  union 42.0   (52 s)


{'clean': 76.9, 'linf': 44.8, 'l2': 62.2, 'l1': 46.2, 'union': 42.0}

## Q3 Check the evaluation

In [53]:
test_model(model_mine2_s1, steps=200, restarts=5)

clean 76.2  linf 44.5  l2 61.3  l1 43.8  union 40.4   (439 s)


{'clean': 76.2, 'linf': 44.5, 'l2': 61.3, 'l1': 43.8, 'union': 40.4}

In [54]:
test_model(model_mine2_s0, steps=200, restarts=5)

clean 76.4  linf 44.3  l2 61.9  l1 43.8  union 40.4   (517 s)


{'clean': 76.4, 'linf': 44.3, 'l2': 61.9, 'l1': 43.8, 'union': 40.4}

In [57]:
test_model(model_mine2_nd_s1, steps=200, restarts=5)

clean 77.2  linf 44.2  l2 62.4  l1 44.3  union 40.5   (438 s)


{'clean': 77.2, 'linf': 44.2, 'l2': 62.4, 'l1': 44.3, 'union': 40.5}

In [58]:
test_model(model_mine2_nd_s0, steps=200, restarts=5)

clean 76.9  linf 44.4  l2 62.2  l1 45.1  union 41.5   (523 s)


{'clean': 76.9, 'linf': 44.4, 'l2': 62.2, 'l1': 45.1, 'union': 41.5}

In [55]:
test_model(model_ramp_s1, steps=200, restarts=5)

clean 78.4  linf 41.6  l2 62.3  l1 44.0  union 39.2   (443 s)


{'clean': 78.4, 'linf': 41.6, 'l2': 62.3, 'l1': 44.0, 'union': 39.2}

In [56]:
test_model(model_ramp_s0, steps=200, restarts=5)

clean 78.5  linf 41.5  l2 62.5  l1 43.5  union 39.1   (518 s)


{'clean': 78.5, 'linf': 41.5, 'l2': 62.5, 'l1': 43.5, 'union': 39.1}